# Single-Qubit GST workflow (plan/simulate → analyze)

It **imports** gate models from `gate_models.py`.

## Expected directory contents
- `gate_models.py`

## Files created by this notebook
- `gst_circuit_plan_example.gstdata`
- `circuit_design.yml`
- `my_gst_outcomes.gstdata`


In [1]:
# Imports
from pathlib import Path
import numpy as np

import ionsim as ism

# Import your gate models (stand-alone module)
from gate_models import (
    X_pi2_process_matrix,
    Y_pi2_process_matrix,
    shuttle_w_dephasing,
    t_Xpi2,
    t_Ypi2,
)


## Step 1 — Create circuit plan and optionally simulate experimental outcomes


In [2]:
# ---- User-tunable configuration (planning + simulation) ----

# Gate-set choice (Option 1 from your script)
gate_names = ['Gxpi2', 'Gypi2', 'shuttle', 'shuttle_inverse']
germ_powers = [1, 2, 4, 8, 16]
germ_sequences = [
    ['Gxpi2'],
    ['Gypi2'],
    ['shuttle', 'shuttle_inverse'],
    ['shuttle', 'Gxpi2', 'shuttle_inverse'],
    ['shuttle', 'Gypi2', 'shuttle_inverse'],
]

qubit_indices = [0]
num_qubits = len(qubit_indices)

# Planning/design file outputs
gst_circuit_filename = './gst_circuit_plan_example.gstdata'
design_file = './circuit_design.yml'

# Outcome file output
circuit_outcome_file = './my_gst_outcomes.gstdata'

# Shot count
N_shots = 500

# Error-model parameters used to simulate data
dephasing_rate = 8000.0        # rad/s
gate_duration = 10e-6          # seconds
dephasing_probability = dephasing_rate * gate_duration

excess_X_rotation = 0.005
excess_Y_rotation = 0.005
shuttling_Z_error = 0.0065


In [3]:
# (A) Write circuit plan + circuit design if needed

if Path(gst_circuit_filename).exists():
    print('GST circuit plan already exists:', gst_circuit_filename)
else:
    print('Writing GST circuit plan to:', gst_circuit_filename)
    planner = ism.GSTCircuitPlanner(
        gate_names,
        qubit_indices,
        germ_powers=germ_powers,
        germs=germ_sequences,
    )
    planner.write_circuit_plan(gst_circuit_filename, num_qubits)
    planner.write_circuit_design(design_file)
    print('Wrote design file to:', design_file)




Writing GST circuit plan to: ./gst_circuit_plan_example.gstdata
Wrote design file to: ./circuit_design.yml


In [4]:
# (B) If not running experiments, read the circuits and simulate outcomes using the gate models
gst_circuits = ism.parse_gst_circuit_file(gst_circuit_filename)
print('Number of circuits:', len(gst_circuits))

# (C) Build mapping from gate name -> 4x4 process matrix
gate_process_matrices = {
    'Gxpi2': X_pi2_process_matrix(excess_X_rotation, dephasing_probability),
    'Gypi2': Y_pi2_process_matrix(excess_Y_rotation, dephasing_probability),
    'shuttle': shuttle_w_dephasing(shuttling_Z_error, dephasing_probability),
    'shuttle_inverse': shuttle_w_dephasing(-shuttling_Z_error, dephasing_probability),
}

# (D) Set up IonSim basis/state for simulating outcomes
spins = [ism.AtomicSpin.from_species(species='171Yb+', term_symbols=['S1/2'], level_names=['S1/2,0,0', 'S1/2,1,0'])]
basis = ism.StandardBasis([*spins])

rho_0 = ism.State.from_coefficients(basis, np.array([1.0, 0.0]))  # |0>
outcome_labels = ['0', '1']

# (E) Create outcomes file and simulate each circuit
ism.GSTCircuitPlanner.create_circuit_outcomes_file(circuit_outcome_file, 1)
print('Writing outcomes to:', circuit_outcome_file)

for idx, circuit in enumerate(gst_circuits):
    rho = rho_0
    for gate in circuit.expanded_gates:
        rho = rho.propagate_using_process_matrix(gate_process_matrices[gate.name])

    outcome_probabilities = rho.compute_basis_state_probabilities()
    counts = np.random.multinomial(N_shots, [outcome_probabilities[0], outcome_probabilities[1]])

    outcome_info = {label: int(c) for label, c in zip(outcome_labels, counts)}
    circuit.measurement_data = ism.CircuitData.from_counts(outcome_info)
    circuit.append_to_file(circuit_outcome_file)

print('Finished. Outcomes file created:', circuit_outcome_file)


Number of circuits: 1354
Writing outcomes to: ./my_gst_outcomes.gstdata
Finished. Outcomes file created: ./my_gst_outcomes.gstdata


## Step 2 — Run GST analysis on the outcomes


In [5]:
import ionsim as sm

def run_GST(fname: str, include_SPAM_error: bool = True):
    # 1. Import GST circuit outcomes
    parsed_circuits = sm.parse_gst_circuit_file(fname)

    # 2. Set up 1Q basis
    spins = [
        sm.AtomicSpin.from_species(
            species='171Yb+',
            term_symbols=['S1/2'],
            level_names=['S1/2,0,0', 'S1/2,1,0'],
        )
    ]
    basis = sm.StandardBasis([*spins])

    # 3. Map GST gate names -> gate-model functions
    ism_gate_models = {
        'Gxpi2': X_pi2_process_matrix,
        'Gypi2': Y_pi2_process_matrix,
        'shuttle': shuttle_w_dephasing,
        'shuttle_inverse': shuttle_w_dephasing,
        # 'tGxpi2': t_Xpi2,
        # 'tGypi2': t_Ypi2,
    }

    # 4. SPAM parametrizations
    ideal_prep_state = sm.State.from_coefficients(basis, [1.0, 0.0])
    d = len(basis.states)

    def prep_state_function(state_parameters):
        prep_state = np.zeros(len(ideal_prep_state.supervector), dtype=complex)
        prep_state += ideal_prep_state.supervector
        prep_state[:-1] += state_parameters

        # Enforce Tr[rho]=1 constraint
        diag_indices = [i * (d + 1) for i in range(d)]
        prep_state[-1] = 1.0 - np.sum(prep_state[diag_indices[:-1]])
        return prep_state

    ideal_POVM_effects = {
        '0': sm.EnergyShiftOperator.from_matrix(basis, sm.Pauli.projector_0),
        '1': sm.EnergyShiftOperator.from_matrix(basis, sm.Pauli.projector_1),
    }
    N_effects = len(ideal_POVM_effects)
    assert N_effects == d

    POVM_models = {}
    # Note: last POVM effect is constrained by completeness and set to None
    for i, (outcome, ideal_effect) in enumerate(ideal_POVM_effects.items()):
        if i == (N_effects - 1):
            POVM_models[outcome] = None
            break

        def effect_function(effect_parameters, POVM_operator=ideal_effect):
            return POVM_operator.superbra + effect_parameters

        POVM_models[outcome] = effect_function

    # 5. Ideal gate set for error metric and linear GST initial guess 
    ideal_gate_set = {
        'Gxpi2': X_pi2_process_matrix(0.0, 0.0),
        'Gypi2': Y_pi2_process_matrix(0.0, 0.0),
        'shuttle': shuttle_w_dephasing(0.0, 0.0),
        'shuttle_inverse': shuttle_w_dephasing(0.0, 0.0),
        'prep': ideal_prep_state,
        'POVM': ideal_POVM_effects,
    }

    # Set dephasing parameter bounds to be positive
    parameter_bounds = {
        'Gxpi2': {'dephasing_probability': (0.0, None)},
        'Gypi2': {'dephasing_probability': (0.0, None)},
        'shuttle': {'dephasing_probability': (0.0, None)},
        'shuttle_inverse': {'dephasing_probability': (0.0, None)},
    }

    # 6. Load circuit design and run GST
    gst_circuit_design = sm.GSTCircuitPlanner.load_design('circuit_design.yml')

    GST_analyzer = sm.GateSetTomography(
        basis,
        prep_state_function,
        POVM_models,
        parsed_circuits,
        ism_gate_models,
        circuit_design=gst_circuit_design,
        ideal_gate_set=ideal_gate_set,
        verbose=False,
        parameter_bounds=parameter_bounds,
    )

    solver_results = GST_analyzer.solve_for_gate_parameters(parameters_guess = None, solver='staged MLE')

    gst_parameter_vector = solver_results.x
    GST_analyzer.print_parameters()
    GST_analyzer.print_state_and_POVMs()

    gate_set_error = GST_analyzer.compute_gate_set_error(
        gst_parameter_vector,
        ideal_gate_set,
        include_SPAM_error=True,
    )
    
    # Access error for each gate in the gate set (model vs. ideal; process infidelity)
    error_by_gate_set_element = GST_analyzer.compute_gate_set_error_by_element(gst_parameter_vector, ideal_gate_set, error_metric = 'process infidelity') 
    print(f"\n\n\nError for each gate set element compared to ideal: \n")
    print(error_by_gate_set_element)  

    return gate_set_error


gate_set_error = run_GST(circuit_outcome_file)
print('\nGate set error (best-fit vs ideal):', gate_set_error)



 Printing gate set: 
Gate: Gypi2:0
Name: Gypi2

Gate: shuttle:0
Name: shuttle

Gate: Gxpi2:0
Name: Gxpi2

Gate: shuttle_inverse:0
Name: shuttle_inverse

 --- Running Maximum likelihood estimation analysis --- 

 --- Printing parameter values --- 
Prep state parameters: [ 0.01731157 -0.00026345 -0.00038959]

Measurement effect parameters: [-0.01658459  0.00593785 -0.00179442  0.01735531]

 Gate Gypi2 parameters: {'excess_Y_rot': np.float64(0.004972218857275409), 'dephasing_probability': np.float64(0.08207281009915099)}

 Gate shuttle parameters: {'Z_error': np.float64(-6.027125406865803e-06), 'dephasing_probability': np.float64(0.08130749194606902)}

 Gate Gxpi2 parameters: {'excess_X_rot': np.float64(0.0034217569628716767), 'dephasing_probability': np.float64(0.08079004466032834)}

 Gate shuttle_inverse parameters: {'Z_error': np.float64(0.013751026426143597), 'dephasing_probability': np.float64(0.07766691924524201)}

Prep state supervector: [ 1.01731157e+00+0.j -2.63450188e-04+0.j -3